In [ ]:
!wget -q https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz
!tar -xzf mmseqs-linux-avx2.tar.gz
!cp mmseqs/bin/mmseqs /usr/local/bin/
!mmseqs version

# Download the two FASTA datasets from GitHub
!wget -q https://raw.githubusercontent.com/sajjadrezvani/SigPep-LB2/main/data/raw/positive.fasta
!wget -q https://raw.githubusercontent.com/sajjadrezvani/SigPep-LB2/main/data/raw/negative.fasta

d401e78c2d18a822cdb1527d7464a043f6035a15


In [ ]:
# 1. Clean workspace
!rm -rf tmp_pos tmp_neg
!mkdir -p tmp_pos tmp_neg

# ==============================================================================
# NEGATIVE DATASET: Explicit connected-component clustering
# ==============================================================================
# Create sequence DB
!mmseqs createdb negative.fasta tmp_neg/neg_db

# Compute all pairwise alignments directly (avoids linclust)
!mmseqs search tmp_neg/neg_db tmp_neg/neg_db tmp_neg/aln tmp_neg/work \
    --min-seq-id 0.3 -c 0.4 --cov-mode 0

# Cluster alignments using connected components (--cluster-mode 1)
!mmseqs clust tmp_neg/neg_db tmp_neg/aln tmp_neg/neg_clu \
    --cluster-mode 1

# Extract representatives to FASTA and TSV
!mmseqs createsubdb tmp_neg/neg_clu tmp_neg/neg_db tmp_neg/neg_rep
!mmseqs convert2fasta tmp_neg/neg_rep neg_clustered_rep_seq.fasta
!mmseqs createtsv tmp_neg/neg_db tmp_neg/neg_db tmp_neg/neg_clu neg_clustered_cluster.tsv

# ==============================================================================
# POSITIVE DATASET: Explicit connected-component clustering
# ==============================================================================
# Create sequence DB
!mmseqs createdb positive.fasta tmp_pos/pos_db

# Compute all pairwise alignments directly
!mmseqs search tmp_pos/pos_db tmp_pos/pos_db tmp_pos/aln tmp_pos/work \
    --min-seq-id 0.3 -c 0.4 --cov-mode 0

# Cluster alignments using connected components (--cluster-mode 1)
!mmseqs clust tmp_pos/pos_db tmp_pos/aln tmp_pos/pos_clu \
    --cluster-mode 1

# Extract representatives to FASTA and TSV
!mmseqs createsubdb tmp_pos/pos_clu tmp_pos/pos_db tmp_pos/pos_rep
!mmseqs convert2fasta tmp_pos/pos_rep pos_clustered_rep_seq.fasta
!mmseqs createtsv tmp_pos/pos_db tmp_pos/pos_db tmp_pos/pos_clu pos_clustered_cluster.tsv

createdb negative.fasta tmp_neg/neg_db 

MMseqs Version:                    	d401e78c2d18a822cdb1527d7464a043f6035a15
Database type                      	0
Shuffle input database             	true
Createdb mode                      	0
Write lookup file                  	1
Offset of numeric ids              	0
Threads                            	2
Compressed                         	0
Mask residues                      	0
Mask residues probability          	0.9
Mask lower case residues           	0
Mask lower letter repeating N times	0
Use GPU                            	0
Verbosity                          	3

Converting sequences
[19494] 0s 30ms
Time for merging to neg_db_h: 0h 0m 0s 4ms
Time for merging to neg_db: 0h 0m 0s 8ms
Database type: Aminoacid
Time for processing: 0h 0m 0s 64ms
Create directory tmp_neg/work
search tmp_neg/neg_db tmp_neg/neg_db tmp_neg/aln tmp_neg/work --min-seq-id 0.3 -c 0.4 --cov-mode 0 

MMseqs Version:                        	d401e78c2d18a822cdb1527d7464a0

In [ ]:
import re
import pandas as pd

def extract_accession(header_str: str) -> str:
    """Extracts standard UniProt accession (e.g., P12345, Q9Y261) or falls back to first token."""
    text = str(header_str).strip().lstrip(">")
    match = re.search(r'([O,P,Q][0-9][A-Z,0-9]{3}[0-9]|[A-N,R-Z][0-9]([A-Z][A-Z,0-9]{2}[0-9]){1,2})', text)
    if match:
        return match.group(1)
    if "|" in text:
        parts = text.split("|")
        return parts[1] if len(parts) > 1 and parts[0] in ("sp", "tr") else parts[0]
    return text.split()[0]

def parse_fasta_lengths(fasta_file: str):
    """Parses FASTA file and returns a mapping of Accession -> Sequence Length."""
    lengths = {}
    current_acc = None
    current_len = 0

    with open(fasta_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_acc:
                    lengths[current_acc] = current_len
                current_acc = extract_accession(line)
                current_len = 0
            else:
                current_len += len(line)
        if current_acc:
            lengths[current_acc] = current_len

    return lengths

def analyze_clusters(cluster_tsv: str, original_fasta: str, top_n: int = 10):
    # MMseqs2 TSV: Column 0 = Representative, Column 1 = Member sequence
    df = pd.read_csv(cluster_tsv, sep="\t", header=None, names=["Raw_Rep", "Raw_Mem"])

    # Extract clean UniProt accessions for matching
    df["Representative"] = df["Raw_Rep"].apply(extract_accession)
    df["Member"] = df["Raw_Mem"].apply(extract_accession)

    # Read sequence lengths from the FASTA file
    seq_lengths = parse_fasta_lengths(original_fasta)

    # Map sequence lengths
    df["Member_Length"] = df["Member"].map(seq_lengths)

    # Check if any mapped length is still missing
    missing_count = df["Member_Length"].isna().sum()
    if missing_count > 0:
        print(f"Warning: {missing_count} sequences could not be matched to the FASTA file.")

    # Aggregate cluster statistics
    cluster_summary = df.groupby("Representative").agg(
        Total_Sequences=("Member", "count"),
        Representative_Length=("Representative", lambda s: seq_lengths.get(s.iloc[0], "N/A")),
        Member_Accessions=("Member", list),
        Member_Lengths=("Member_Length", list)
    ).sort_values(by="Total_Sequences", ascending=False).reset_index()

    print(f"=== CLUSTERING ANALYSIS: {cluster_tsv} ===")
    print(f"Total Clusters: {len(cluster_summary)}\n")

    top_df = cluster_summary.head(top_n)
    for idx, row in top_df.iterrows():
        print(f"Rank {idx+1}:")
        print(f"  Representative ID     : {row['Representative']}")
        print(f"  Representative Length : {row['Representative_Length']} aa")
        print(f"  Cluster Size          : {row['Total_Sequences']} sequences")
        print(f"  Member IDs            : {row['Member_Accessions']}")
        print(f"  Member Lengths (aa)   : {row['Member_Lengths']}\n")

    return cluster_summary

# Run for Positive dataset
pos_summary = analyze_clusters("pos_clustered_cluster.tsv", "positive.fasta", top_n=10)

# Run for Negative dataset
neg_summary = analyze_clusters("neg_clustered_cluster.tsv", "negative.fasta", top_n=10)

=== CLUSTERING ANALYSIS: pos_clustered_cluster.tsv ===
Total Clusters: 1062

Rank 1:
  Representative ID     : Q90249
  Representative Length : 137 aa
  Cluster Size          : 102 sequences
  Member IDs            : ['Q90249', 'P45881', 'P49121', 'P24605', 'Q8AXY1', 'Q1ZY03', 'P00624', 'Q90W39', 'Q8UVU7', 'A6MEY4', 'G9I930', 'A8CG89', 'Q2PG81', 'P00593', 'A0A411EZW9', 'P06859', 'Q02471', 'P34180', 'Q2YHJ4', 'P70090', 'P31100', 'Q1RP79', 'Q2YHJ9', 'B5U6Y4', 'P0CAR9', 'P82114', 'A8CG78', 'B5U6Z2', 'P0CAS0', 'Q9PVF4', 'Q3HLQ4', 'P23028', 'P80963', 'Q8UVZ7', 'P14555', 'A8CG82', 'Q2YHJ6', 'D0UGJ0', 'A8CG86', 'P0DJN6', 'P0DJJ9', 'C0HLL2', 'P31854', 'P14424', 'P0DJN7', 'F8QN54', 'Q6EER4', 'Q6H3D2', 'Q6H3D3', 'P17935', 'Q2YHJ3', 'Q6JK69', 'Q6H3D5', 'Q2YHJ7', 'A8CG90', 'Q2YHJ2', 'A8E2V8', 'P0DJJ8', 'Q4VRI5', 'Q1RP78', 'Q7T2R1', 'Q2YHJ8', 'P20474', 'A4VBF0', 'C0HKC2', 'Q8QG87', 'P08878', 'C0HKC3', 'Q805A2', 'Q7ZTA6', 'P00592', 'F8QN53', 'P04054', 'P00625', 'P24027', 'Q2HZ28', 'Q90Y77', 'Q6H3C7'

In [ ]:
def show_bottom_clusters(cluster_summary_df, dataset_name: str, bottom_n: int = 10):
    print(f"=== 10 SMALLEST / BOTTOM-RANKED CLUSTERS: {dataset_name} ===")
    bottom_df = cluster_summary_df.tail(bottom_n).iloc[::-1].reset_index(drop=True)

    for idx, row in bottom_df.iterrows():
        print(f"\nBottom Rank {idx + 1}:")
        print(f"  Representative ID     : {row['Representative']}")
        print(f"  Representative Length : {row['Representative_Length']} aa")
        print(f"  Cluster Size          : {row['Total_Sequences']} sequence(s)")
        print(f"  Member IDs            : {row['Member_Accessions']}")
        print(f"  Member Lengths (aa)   : {row['Member_Lengths']}")

# Display bottom 10 for positive dataset
show_bottom_clusters(pos_summary, "POSITIVE DATASET", bottom_n=10)

# Display bottom 10 for negative dataset
show_bottom_clusters(neg_summary, "NEGATIVE DATASET", bottom_n=10)

=== 10 SMALLEST / BOTTOM-RANKED CLUSTERS: POSITIVE DATASET ===

Bottom Rank 1:
  Representative ID     : Q9Y2I2
  Representative Length : 539 aa
  Cluster Size          : 1 sequence(s)
  Member IDs            : ['Q9Y2I2']
  Member Lengths (aa)   : [539]

Bottom Rank 2:
  Representative ID     : Q9Y4X3
  Representative Length : 112 aa
  Cluster Size          : 1 sequence(s)
  Member IDs            : ['Q9Y4X3']
  Member Lengths (aa)   : [112]

Bottom Rank 3:
  Representative ID     : Q9Y5Q6
  Representative Length : 135 aa
  Cluster Size          : 1 sequence(s)
  Member IDs            : ['Q9Y5Q6']
  Member Lengths (aa)   : [135]

Bottom Rank 4:
  Representative ID     : Q9Y5U5
  Representative Length : 241 aa
  Cluster Size          : 1 sequence(s)
  Member IDs            : ['Q9Y5U5']
  Member Lengths (aa)   : [241]

Bottom Rank 5:
  Representative ID     : Q9Y6C2
  Representative Length : 1016 aa
  Cluster Size          : 1 sequence(s)
  Member IDs            : ['Q9Y6C2']
  Member Leng

In [ ]:
import csv
import re
import random
from typing import Set, Dict, List, Tuple

# ---------------------------------------------------------------------------
# 1. Helper: Extract accessions from FASTA
# ---------------------------------------------------------------------------
def get_rep_accessions(fasta_file: str) -> Set[str]:
    reps = set()
    with open(fasta_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith(">"):
                match = re.search(r'([O,P,Q][0-9][A-Z,0-9]{3}[0-9]|[A-N,R-Z][0-9]([A-Z][A-Z,0-9]{2}[0-9]){1,2})', line)
                if match:
                    reps.add(match.group(1).strip())
                else:
                    reps.add(line[1:].strip().split()[0].split("|")[0])
    return reps

# ---------------------------------------------------------------------------
# 2. Helper: Load & filter TSV by representative accessions
# ---------------------------------------------------------------------------
def load_and_filter_tsv(tsv_file: str, valid_reps: Set[str]) -> List[Dict[str, str]]:
    filtered = []
    with open(tsv_file, "r", encoding="utf-8") as f:
        first_line = f.readline()
        f.seek(0)
        delimiter = "\t" if "\t" in first_line else ","
        reader = csv.DictReader(f, delimiter=delimiter)
        reader.fieldnames = [k.strip() for k in reader.fieldnames] if reader.fieldnames else []

        acc_col = None
        for col in reader.fieldnames:
            if any(term in col.lower() for term in ["acc", "entry", "id"]):
                acc_col = col
                break
        if not acc_col:
            acc_col = reader.fieldnames[0]

        for row in reader:
            raw_acc = row[acc_col].strip()
            clean_acc = raw_acc.split("|")[1] if "|" in raw_acc else raw_acc

            if clean_acc in valid_reps or raw_acc in valid_reps:
                row["Accession"] = clean_acc
                filtered.append(row)
    return filtered

# ---------------------------------------------------------------------------
# 3. Helper: Perform independent 80/20 partition
# ---------------------------------------------------------------------------
def split_dataset(rows: List[Dict[str, str]], seed: int = 42) -> Tuple[List[Dict[str, str]], List[Dict[str, str]]]:
    random.seed(seed)
    shuffled = rows.copy()
    random.shuffle(shuffled)

    n_train = int(0.8 * len(shuffled))
    train_part = shuffled[:n_train]
    bench_part = shuffled[n_train:]

    return train_part, bench_part

def write_tsv(rows: List[Dict[str, str]], filename: str):
    if not rows:
        print(f"Warning: {filename} has 0 rows.")
        return
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys(), delimiter="\t")
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved: {filename} ({len(rows)} entries)")

# ---------------------------------------------------------------------------
# 4. Helper: Export corresponding partitioned FASTA files
# ---------------------------------------------------------------------------
def split_fasta(rep_fasta: str,
                train_accs: Set[str],
                bench_accs: Set[str],
                train_fa_out: str,
                bench_fa_out: str):
    with open(rep_fasta, "r", encoding="utf-8") as in_fa, \
         open(train_fa_out, "w", encoding="utf-8") as tr_out, \
         open(bench_fa_out, "w", encoding="utf-8") as be_out:

        current_target = None
        for line in in_fa:
            if line.startswith(">"):
                match = re.search(r'([O,P,Q][0-9][A-Z,0-9]{3}[0-9]|[A-N,R-Z][0-9]([A-Z][A-Z,0-9]{2}[0-9]){1,2})', line)
                acc = match.group(1).strip() if match else line[1:].strip().split()[0].split("|")[0]

                if acc in train_accs:
                    current_target = tr_out
                elif acc in bench_accs:
                    current_target = be_out
                else:
                    current_target = None

            if current_target:
                current_target.write(line)

# ===========================================================================
# Execution
# ===========================================================================
# 1. Parse representative IDs
pos_reps = get_rep_accessions("pos_clustered_rep_seq.fasta")
neg_reps = get_rep_accessions("neg_clustered_rep_seq.fasta")

# 2. Filter TSVs
pos_nr = load_and_filter_tsv("positive.tsv", pos_reps)
neg_nr = load_and_filter_tsv("negative.tsv", neg_reps)

# 3. Independent 80/20 Splits on TSVs
pos_train, pos_bench = split_dataset(pos_nr, seed=42)
neg_train, neg_bench = split_dataset(neg_nr, seed=42)

# Save separate TSV files
write_tsv(pos_train, "positive_train.tsv")
write_tsv(pos_bench, "positive_benchmark.tsv")
write_tsv(neg_train, "negative_train.tsv")
write_tsv(neg_bench, "negative_benchmark.tsv")

# 4. Independent 80/20 Splits on FASTA files
pos_tr_accs = {r["Accession"] for r in pos_train}
pos_be_accs = {r["Accession"] for r in pos_bench}
split_fasta("pos_clustered_rep_seq.fasta", pos_tr_accs, pos_be_accs, "positive_train.fasta", "positive_benchmark.fasta")

neg_tr_accs = {r["Accession"] for r in neg_train}
neg_be_accs = {r["Accession"] for r in neg_bench}
split_fasta("neg_clustered_rep_seq.fasta", neg_tr_accs, neg_be_accs, "negative_train.fasta", "negative_benchmark.fasta")

print("\nProcessing complete. All positive and negative train/benchmark sets are created.")

Saved: positive_train.tsv (849 entries)
Saved: positive_benchmark.tsv (213 entries)
Saved: negative_train.tsv (7017 entries)
Saved: negative_benchmark.tsv (1755 entries)

Processing complete. All positive and negative train/benchmark sets are created.


In [ ]:
import csv
import random
from typing import List, Dict

def assign_and_record_cv_folds(input_tsv: str,
                               output_tsv: str,
                               label: int,
                               n_folds: int = 5,
                               seed: int = 42) -> List[Dict[str, str]]:
    with open(input_tsv, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        rows = list(reader)
        fieldnames = list(reader.fieldnames) if reader.fieldnames else []

    random.seed(seed)
    random.shuffle(rows)

    for idx, row in enumerate(rows):
        row["Label"] = str(label)
        row["Set"] = "Training"
        row["CV_Fold"] = str(idx % n_folds)

    for col in ["Label", "Set", "CV_Fold"]:
        if col not in fieldnames:
            fieldnames.append(col)

    with open(output_tsv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t", extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

    print(f"Saved: {output_tsv} ({len(rows)} proteins partitioned into {n_folds} folds)")
    return rows

# 1. Process positive training set
pos_cv_data = assign_and_record_cv_folds(
    input_tsv="positive_train.tsv",
    output_tsv="positive_train_5fold.tsv",
    label=1,
    n_folds=5,
    seed=42
)

# 2. Process negative training set
neg_cv_data = assign_and_record_cv_folds(
    input_tsv="negative_train.tsv",
    output_tsv="negative_train_5fold.tsv",
    label=0,
    n_folds=5,
    seed=42
)

# 3. Combine both and gather all unique keys across all rows
master_cv_rows = pos_cv_data + neg_cv_data

# Preserve order while gathering all unique columns from both sets
master_fields = []
for row in master_cv_rows:
    for k in row.keys():
        if k not in master_fields:
            master_fields.append(k)

# Write master CV table
with open("training_5fold_cv_master.tsv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=master_fields, delimiter="\t", restval="", extrasaction="ignore")
    writer.writeheader()
    writer.writerows(master_cv_rows)

print(f"Saved combined master CV file: training_5fold_cv_master.tsv ({len(master_cv_rows)} total rows)")

Saved: positive_train_5fold.tsv (849 proteins partitioned into 5 folds)
Saved: negative_train_5fold.tsv (7017 proteins partitioned into 5 folds)
Saved combined master CV file: training_5fold_cv_master.tsv (7866 total rows)


In [ ]:
import pandas as pd

df = pd.read_csv("training_5fold_cv_master.tsv", sep="\t")
summary_table = pd.crosstab(df["CV_Fold"], df["Label"], margins=True)
summary_table.columns = ["Negative (0)", "Positive (1)", "Total"]
print(summary_table)

         Negative (0)  Positive (1)  Total
CV_Fold                                   
0                1404           170   1574
1                1404           170   1574
2                1403           170   1573
3                1403           170   1573
4                1403           169   1572
All              7017           849   7866


In [ ]:
import pandas as pd

df = pd.read_csv("training_5fold_cv_master.tsv", sep="\t")

# 1. Check contingency table across folds and labels
contingency = pd.crosstab(df["CV_Fold"], df["Label"], margins=True)
contingency.columns = ["Negative (0)", "Positive (1)", "Total"]

# 2. Compute ratio of Negatives to Positives per fold
ratio_per_fold = (contingency["Negative (0)"] / contingency["Positive (1)"]).round(2)
contingency["Neg:Pos Ratio"] = ratio_per_fold

print("=== 5-FOLD CROSS-VALIDATION BALANCE CHECK ===")
print(contingency)

# 3. Check for unexpected null values in fold assignments
missing_folds = df["CV_Fold"].isna().sum()
print(f"\nMissing fold assignments: {missing_folds}")

=== 5-FOLD CROSS-VALIDATION BALANCE CHECK ===
         Negative (0)  Positive (1)  Total  Neg:Pos Ratio
CV_Fold                                                  
0                1404           170   1574           8.26
1                1404           170   1574           8.26
2                1403           170   1573           8.25
3                1403           170   1573           8.25
4                1403           169   1572           8.30
All              7017           849   7866           8.27

Missing fold assignments: 0


In [ ]:
df

,accession,organism,kingdom,protein_length,cleavage_position,Accession,Label,Set,CV_Fold,tm_helix_first_90
0,P67861,Daboia russelii,Metazoa,144,24.0,P67861,1,Training,0,NaN
1,P80961,Myoxocephalus octodecemspinosus,Metazoa,128,20.0,P80961,1,Training,1,NaN
2,Q16927,Aedes aegypti,Metazoa,2169,46.0,Q16927,1,Training,2,NaN
3,Q7TQN3,Mus musculus,Metazoa,571,29.0,Q7TQN3,1,Training,3,NaN
4,H6U1I8,Tanacetum cinerariifolium,Plants,365,27.0,H6U1I8,1,Training,4,NaN
...,...,...,...,...,...,...,...,...,...,...
7861,Q689G9,Oryza sativa subsp. japonica,Plants,518,NaN,Q689G9,0,Training,2,False
7862,P32776,Saccharomyces cerevisiae (strain ATCC 204508 /...,Fungi,642,NaN,P32776,0,Training,3,False
7863,Q9HCI7,Homo sapiens,Metazoa,577,NaN,Q9HCI7,0,Training,4,False
7864,P69447,Spinacia oleracea,Plants,81,NaN,P69447,0,Training,0,True
